# F1 Pit Stops Prediction — XGBoost Baseline
**Kaggle Playground S6E5 · Metric: ROC-AUC**

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

## Data

In [ ]:
train = pd.read_csv('F1 Pit Stops Series S6E5/train.csv')
test  = pd.read_csv('F1 Pit Stops Series S6E5/test.csv')

X, y = train.drop(['id', 'PitNextLap'], axis=1), train['PitNextLap']
X_test, test_ids = test.drop('id', axis=1), test['id'].values

print(f"Train {X.shape} | Test {X_test.shape} | Pit rate {y.mean():.2%}")

## EDA

Two quick checks before engineering: class balance and the top numeric correlations with the target.

In [ ]:
# Class balance
sns.countplot(x=y, palette=['#1F1F27', '#E10600'])
plt.xticks([0, 1], ['No Pit', 'Pit Next Lap'])
plt.title('Class balance (~20% positive)')
plt.tight_layout(); plt.show()

# Top correlations with target
corr = (pd.concat([X.select_dtypes(include=np.number), y], axis=1)
          .corr()['PitNextLap']
          .drop('PitNextLap')
          .sort_values(key=abs, ascending=False)
          .head(10))
corr.plot(kind='barh', color=['#E10600' if v > 0 else '#1F1F27' for v in corr])
plt.title('Top 10 features by correlation with PitNextLap')
plt.tight_layout(); plt.show()

## Feature Engineering

Three feature families:
- **Tyre degradation** — polynomial and log transforms of `TyreLife`, plus a compound-weighted urgency score
- **Race context interactions** — `TyreLife` crossed with `Stint`, `RaceProgress`, and `LapNumber`
- **Lap time signals** — degradation per lap, lap-time consistency, absolute delta

In [ ]:
COMPOUND_DEG = {'SOFT': 3, 'MEDIUM': 2, 'HARD': 1}

def engineer_features(df):
    df = df.copy()

    # Tyre degradation
    df['TyreLife_Sq']  = df['TyreLife'] ** 2
    df['TyreLife_Log'] = np.log1p(df['TyreLife'])
    df['TyreLife_Bin'] = pd.cut(df['TyreLife'], bins=[0, 10, 20, 30, 50],
                                 labels=[0, 1, 2, 3]).astype(float)

    df['Compound_DegRate'] = df['Compound'].map(COMPOUND_DEG).fillna(2)
    df['Pit_Urgency']      = df['TyreLife'] * df['Compound_DegRate'] * (1 + df['RaceProgress'])
    df['Stint_Age']        = df['Stint'] + df['TyreLife'] / 30

    # Race-context interactions
    df['Stint_x_TyreLife']    = df['Stint'] * df['TyreLife']
    df['Progress_x_TyreLife'] = df['RaceProgress'] * df['TyreLife']
    df['Lap_x_TyreLife']      = df['LapNumber'] * df['TyreLife']
    df['Early_Race']           = (df['RaceProgress'] < 0.3).astype(float)
    df['Mid_Race']             = df['RaceProgress'].between(0.3, 0.7).astype(float)
    df['Late_Race']            = (df['RaceProgress'] >= 0.7).astype(float)

    # Lap time signals
    df['Degradation_per_Lap']   = df['Cumulative_Degradation'] / (df['LapNumber'] + 1)
    df['LapTime_Deg']           = df['LapTime (s)'] * df['Cumulative_Degradation'].abs()
    df['LapTime_Consistency']   = 1 / (1 + df['LapTime_Delta'].abs())
    df['LapTime_Delta_x_Life']  = df['LapTime_Delta'] * df['TyreLife']
    df['Position_x_TyreLife']   = df['Position'] * df['TyreLife']

    return df.replace([np.inf, -np.inf], np.nan)

X_eng      = engineer_features(X)
X_test_eng = engineer_features(X_test)
print(f"Added {X_eng.shape[1] - X.shape[1]} engineered features")

## Preprocessing

Target-encode the three categorical columns using training-set means, then median-impute everything.

In [ ]:
CAT_COLS = ['Driver', 'Compound', 'Race']
global_mean = y.mean()

def target_encode(X_tr, X_te, target):
    X_tr, X_te = X_tr.copy(), X_te.copy()
    for col in CAT_COLS:
        means = target.groupby(X_tr[col]).mean()
        X_tr[col] = X_tr[col].map(means).fillna(global_mean)
        X_te[col] = X_te[col].map(means).fillna(global_mean)
    return X_tr, X_te

X_proc, X_test_proc = target_encode(X_eng, X_test_eng, y)

imputer = SimpleImputer(strategy='median')
X_proc      = pd.DataFrame(imputer.fit_transform(X_proc),      columns=X_proc.columns)
X_test_proc = pd.DataFrame(imputer.transform(X_test_proc),     columns=X_test_proc.columns)

print(f"Final feature matrix: {X_proc.shape}")

## Model

`XGBClassifier` with `scale_pos_weight` to handle the class imbalance. Early stopping on a held-out 20% split.

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_proc, y, test_size=0.2, stratify=y, random_state=42
)

scale_pw = (y_tr == 0).sum() / (y_tr == 1).sum()

model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.08,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=0.5,
    reg_alpha=0.3,
    min_child_weight=5,
    scale_pos_weight=scale_pw,
    tree_method='hist',
    early_stopping_rounds=50,
    eval_metric='auc',
    random_state=42,
)

model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=50)

val_auc = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])
print(f"Validation ROC-AUC: {val_auc:.4f}")

## Submission

In [ ]:
preds = model.predict_proba(X_test_proc)[:, 1]

submission = pd.DataFrame({'id': test_ids, 'PitNextLap': preds})
submission.to_csv('submission.csv', index=False)

print(f"Saved {len(submission)} rows | mean P(pit) = {preds.mean():.4f}")
submission.head()